In [2]:
import os
import shutil
import kagglehub
from pathlib import Path

/home/LordAguaKate/Documentos/GitHub/EntrenamientoIA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Definir rutas relativas a partir del directorio base
BASE_DIR = Path("..") 
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
MODELS_DIR = BASE_DIR / "models"

In [4]:
# Descargar el dataset (se descarga en cache primero)
print("Descargando dataset...")
cache_path = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")

dataset_name = "waste_classification"
local_dataset_path = DATA_RAW_DIR / dataset_name

if not local_dataset_path.exists():
    print(f"Moviendo datos a {local_dataset_path}...")
    shutil.copytree(cache_path, local_dataset_path)
else:
    print(f"Los datos ya existen en {local_dataset_path}")

# IMPORTANTE: Verificar la estructura interna del dataset
images_path = local_dataset_path
if (images_path / "images").exists():
    images_path = images_path / "images"
    
print(f"Ruta final de imágenes para entrenamiento: {images_path}")

Descargando dataset...
Moviendo datos a ../data/raw/waste_classification...
Ruta final de imágenes para entrenamiento: ../data/raw/waste_classification/images


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

2026-01-22 11:44:10.146641: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-22 11:44:10.201686: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-22 11:44:11.671947: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [10]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16 

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    validation_split=0.2
)

# Cargar datos de entrenamiento
train_gen = datagen.flow_from_directory(
    str(images_path), # Usamos la ruta corregida del Paso 1
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Cargar datos de validación
val_gen = datagen.flow_from_directory(
    str(images_path),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

NUM_CLASSES = train_gen.num_classes
print(f"Clases encontradas: {NUM_CLASSES}")
print(f"Mapeo de clases: {train_gen.class_indices}")

Found 12000 images belonging to 30 classes.
Found 3000 images belonging to 30 classes.
Clases encontradas: 30
Mapeo de clases: {'aerosol_cans': 0, 'aluminum_food_cans': 1, 'aluminum_soda_cans': 2, 'cardboard_boxes': 3, 'cardboard_packaging': 4, 'clothing': 5, 'coffee_grounds': 6, 'disposable_plastic_cutlery': 7, 'eggshells': 8, 'food_waste': 9, 'glass_beverage_bottles': 10, 'glass_cosmetic_containers': 11, 'glass_food_jars': 12, 'magazines': 13, 'newspaper': 14, 'office_paper': 15, 'paper_cups': 16, 'plastic_cup_lids': 17, 'plastic_detergent_bottles': 18, 'plastic_food_containers': 19, 'plastic_shopping_bags': 20, 'plastic_soda_bottles': 21, 'plastic_straws': 22, 'plastic_trash_bags': 23, 'plastic_water_bottles': 24, 'shoes': 25, 'steel_food_cans': 26, 'styrofoam_cups': 27, 'styrofoam_food_containers': 28, 'tea_bags': 29}


In [12]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [13]:
# Configurar el modelo base pre-entrenado (MobileNetV2)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Congelamos el modelo base para no dañar lo que ya sabe aprender
base_model.trainable = False

print("Modelo base cargado correctamente.")

2026-01-22 12:01:30.462016: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 25s 3us/step
Modelo base cargado correctamente.


In [14]:
# Construcción de la nueva cabecera del modelo
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) # Capa intermedia para aprender patrones complejos
x = Dropout(0.3)(x) # Apaga el 30% de neuronas al azar para evitar memorización (overfitting)

# Capa de salida: NUM_CLASSES debe ser 30 (automático gracias al paso anterior)
output = Dense(NUM_CLASSES, activation='softmax')(x)

# Ensamblamos el modelo final
model = Model(inputs=base_model.input, outputs=output)

print("Arquitectura del modelo finalizada.")
model.summary() # Muestra un resumen para verificar que todo conecta bien

Arquitectura del modelo finalizada.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,425,822 (9.25 MB)

 Trainable params: 167,838 (655.62 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
# Compilación del modelo
model.compile(
    optimizer=Adam(learning_rate=1e-3), # Learning rate estándar para empezar
    loss='categorical_crossentropy',    # La función de pérdida correcta para >2 clases
    metrics=['accuracy']
)

print("Iniciando entrenamiento...")

# Entrenamiento
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15 
)

Iniciando entrenamiento...
Epoch 1/15
  1/750 ━━━━━━━━━━━━━━━━━━━━ 47:10 4s/step - accuracy: 0.0625 - loss: 3.1201

2026-01-22 12:03:57.597450: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 77070336 exceeds 10% of free system memory.
2026-01-22 12:03:57.625867: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 78452736 exceeds 10% of free system memory.
2026-01-22 12:03:57.833914: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 77070336 exceeds 10% of free system memory.
2026-01-22 12:03:57.879743: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 78452736 exceeds 10% of free system memory.


  2/750 ━━━━━━━━━━━━━━━━━━━━ 3:20 268ms/step - accuracy: 0.0781 - loss: 3.1855

2026-01-22 12:03:58.098034: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 77070336 exceeds 10% of free system memory.


750/750 ━━━━━━━━━━━━━━━━━━━━ 245s 323ms/step - accuracy: 0.5139 - loss: 1.6926 - val_accuracy: 0.6910 - val_loss: 0.9670
Epoch 2/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 254s 339ms/step - accuracy: 0.6410 - loss: 1.1841 - val_accuracy: 0.7243 - val_loss: 0.8540
Epoch 3/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 284s 379ms/step - accuracy: 0.6762 - loss: 1.0403 - val_accuracy: 0.7420 - val_loss: 0.8069
Epoch 4/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 266s 354ms/step - accuracy: 0.6901 - loss: 0.9722 - val_accuracy: 0.7647 - val_loss: 0.7370
Epoch 5/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 267s 356ms/step - accuracy: 0.7143 - loss: 0.9022 - val_accuracy: 0.7603 - val_loss: 0.7263
Epoch 6/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 260s 347ms/step - accuracy: 0.7215 - loss: 0.8671 - val_accuracy: 0.7707 - val_loss: 0.6908
Epoch 7/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 274s 365ms/step - accuracy: 0.7398 - loss: 0.8191 - val_accuracy: 0.7723 - val_loss: 0.7032
Epoch 8/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 273s 363ms/step - accuracy: 0.7352 - loss: 0.80

In [16]:
base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.7)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20
)

Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 324s 420ms/step - accuracy: 0.6095 - loss: 1.3705 - val_accuracy: 0.7767 - val_loss: 0.7773
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 313s 417ms/step - accuracy: 0.6883 - loss: 1.0189 - val_accuracy: 0.7887 - val_loss: 0.6720
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 331s 441ms/step - accuracy: 0.7085 - loss: 0.9085 - val_accuracy: 0.7910 - val_loss: 0.6535
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 325s 433ms/step - accuracy: 0.7288 - loss: 0.8476 - val_accuracy: 0.8067 - val_loss: 0.6009
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 340s 453ms/step - accuracy: 0.7394 - loss: 0.8197 - val_accuracy: 0.8043 - val_loss: 0.5998
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 311s 414ms/step - accuracy: 0.7472 - loss: 0.7739 - val_accuracy: 0.8093 - val_loss: 0.5827
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 337s 450ms/step - accuracy: 0.7621 - loss: 0.7297 - val_accuracy: 0.8243 - val_loss: 0.5421
Epoch 8/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 317s 422ms/step - accuracy: 0.7695 -

In [17]:
# Guardar el modelo en formato .keras 
model_save_path = MODELS_DIR / "modelo_residuos_rpi.keras"
model.save(model_save_path)
print(f"Modelo guardado en: {model_save_path}")

Modelo guardado en: ../models/modelo_residuos_rpi.keras


In [18]:
# Conversión a TFLite (Opcional, para tu Raspberry Pi)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_save_path = MODELS_DIR / "modelo_residuos_rpi.tflite"
with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)
print(f"Modelo TFLite guardado en: {tflite_save_path}")

INFO:tensorflow:Assets written to: /tmp/tmp7ngpcme8/assets


INFO:tensorflow:Assets written to: /tmp/tmp7ngpcme8/assets


Saved artifact at '/tmp/tmp7ngpcme8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  139682282137424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282138384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282140880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282140496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282137232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282141072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282139536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282141648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282141264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139682282139344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1396822821376

W0000 00:00:1769121470.069584    9813 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1769121470.069613    9813 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-01-22 16:37:50.072415: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp7ngpcme8
2026-01-22 16:37:50.084273: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-01-22 16:37:50.084296: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp7ngpcme8
I0000 00:00:1769121470.178205    9813 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-01-22 16:37:50.204320: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-01-22 16:37:50.847540: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp7ngpcme8
2026-01-22 16:37:51.029802: I tensorflow/cc/saved_model/loader.cc:471] SavedModel 